# Advanced Problems with Solutions: Decorator Classes, Descriptors, and Method Binding

This notebook is a problem-driven deep dive into **class-based decorators** in Python, especially the subtle interaction between:

- callable objects and `__call__`
- functions as descriptors
- method binding and `__get__`
- `types.MethodType`
- metadata preservation with `functools.update_wrapper`
- parameterized decorator classes
- stateful decorators
- instance-specific state with `weakref.WeakKeyDictionary`
- decorator ordering with `@classmethod` / `@staticmethod`
- composability, exception handling, retries, caching, and re-entrancy

Every problem includes a complete solution and executable checks.

## Learning goals

By the end, you should be able to:

1. Explain why a callable decorator object does **not** automatically behave like a function descriptor.
2. Implement `__get__` correctly for decorated instance methods.
3. Preserve function metadata and signatures.
4. Design decorator classes with configuration and state.
5. Keep state global-per-function or local-per-instance intentionally.
6. Make class-based decorators work with instance, class, and static methods.
7. Write decorators that remain composable and testable.

In [1]:
from __future__ import annotations

from collections import OrderedDict
from functools import update_wrapper
from inspect import signature
from threading import RLock
from time import perf_counter
from types import MethodType
from weakref import WeakKeyDictionary

print("Environment ready.")

Environment ready.


## Warm-up: the core model

A decorator class usually stores the decorated function in `__init__` and implements `__call__` so the resulting instance is callable.

The important distinction is:

```python
@SomeDecorator

def f(...): ...
```

is equivalent to:

```python
f = SomeDecorator(f)
```

So `f` is no longer necessarily a function. It may be an arbitrary callable object.

In [2]:
class BasicLogger:
    def __init__(self, fn):
        self.fn = fn

    def __call__(self, *args, **kwargs):
        print(f"LOG: {self.fn.__name__} called")
        return self.fn(*args, **kwargs)

@BasicLogger
def add(a, b):
    return a + b

assert add(2, 3) == 5
assert isinstance(add, BasicLogger)
print(type(add).__name__)

LOG: add called
BasicLogger


# Problem 1 — Diagnose the missing `self`

The following class-based decorator works for ordinary functions but fails on instance methods.

**Tasks**

1. Reproduce the failure.
2. Explain why `self` is missing.
3. Inspect what `obj.method` actually is before the fix.

In [3]:
class BrokenLogger:
    def __init__(self, fn):
        self.fn = fn

    def __call__(self, *args, **kwargs):
        return self.fn(*args, **kwargs)

class BrokenPerson:
    def __init__(self, name):
        self.name = name

    @BrokenLogger
    def greet(self):
        return f"Hello from {self.name}"

bp = BrokenPerson("Ada")
print("Attribute type:", type(bp.greet).__name__)

try:
    bp.greet()
except TypeError as exc:
    print("Expected failure:", exc)

Attribute type: BrokenLogger
Expected failure: BrokenPerson.greet() missing 1 required positional argument: 'self'


### Solution 1

Normal Python functions implement the descriptor protocol. When a function is retrieved through an instance, its `__get__` creates a **bound method** that automatically supplies the instance as the first argument.

`BrokenLogger` replaces the function with a plain callable instance. Because that instance has no suitable `__get__`, normal method binding never happens. Therefore `bp.greet()` calls the decorator object with no `bp` argument, and the original function receives no `self`.

In [4]:
assert isinstance(BrokenPerson.__dict__["greet"], BrokenLogger)
assert bp.greet is BrokenPerson.__dict__["greet"]
print("The class attribute and instance attribute resolve to the same callable object.")

The class attribute and instance attribute resolve to the same callable object.


# Problem 2 — Implement descriptor binding with `__get__`

Create a class-based decorator that works for both ordinary functions and instance methods.

**Requirements**

- Store the wrapped function.
- Implement `__call__`.
- Implement `__get__(self, instance, owner)`.
- Return the decorator itself when accessed through the class.
- Return a method bound to the instance when accessed through an object.

In [5]:
class MethodAwareLogger:
    def __init__(self, fn):
        self.fn = fn

    def __call__(self, *args, **kwargs):
        print(f"LOG: {self.fn.__name__}")
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

class Person:
    def __init__(self, name):
        self.name = name

    @MethodAwareLogger
    def greet(self, punctuation="!"):
        return f"Hello from {self.name}{punctuation}"

p = Person("Grace")
assert p.greet("!!!") == "Hello from Grace!!!"
assert Person.greet is Person.__dict__["greet"]
print(p.greet)

LOG: greet
<bound method ? of <__main__.Person object at 0x000001AB6A3BEE40>>


### Solution 2: why `MethodType(self, instance)` works

The decorator object itself is callable because it implements `__call__`. `MethodType(self, instance)` therefore creates a bound method whose callable target is the decorator object. When invoked, Python supplies `instance` as the first positional argument to `MethodAwareLogger.__call__`, which forwards it to the original function.

In [6]:
bound = p.greet
print("Bound object:", bound)
print("Bound instance:", bound.__self__)
print("Underlying callable:", bound.__func__)
assert bound.__self__ is p
assert bound.__func__ is Person.__dict__["greet"]

Bound object: <bound method ? of <__main__.Person object at 0x000001AB6A3BEE40>>
Bound instance: <__main__.Person object at 0x000001AB6A3BEE40>
Underlying callable: <__main__.MethodAwareLogger object at 0x000001AB6A3BECF0>


# Problem 3 — Preserve metadata and the original signature

A production-quality decorator should preserve useful metadata such as `__name__`, `__doc__`, `__module__`, `__annotations__`, and `__wrapped__`.

Implement a descriptor-aware decorator that uses `functools.update_wrapper`.

In [7]:
class Logged:
    def __init__(self, fn):
        self.fn = fn
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        print(f"Calling {self.__name__}")
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

@Logged
def power(base: float, exponent: int = 2) -> float:
    """Raise base to exponent."""
    return base ** exponent

assert power.__name__ == "power"
assert power.__doc__ == "Raise base to exponent."
assert str(signature(power)) == "(base: 'float', exponent: 'int' = 2) -> 'float'"
assert power(3, 3) == 27
print(power.__name__, signature(power))

Calling power
power (base: 'float', exponent: 'int' = 2) -> 'float'


### Solution 3: best-practice note

For function decorators implemented as nested functions, `functools.wraps` is common. For callable **instances**, `update_wrapper(self, fn)` is the natural equivalent because it copies metadata onto the instance and sets `__wrapped__`, allowing tools such as `inspect.signature` to follow the wrapper chain.

# Problem 4 — Count calls globally per decorated function

Create a class-based decorator that counts how many times a function or method has been called.

The counter should belong to the **decorator instance**, so all instances of a class share the same count for that decorated method.

In [8]:
class CountCalls:
    def __init__(self, fn):
        self.fn = fn
        self.calls = 0
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

class Calculator:
    @CountCalls
    def double(self, x):
        return x * 2

c1 = Calculator()
c2 = Calculator()
assert c1.double(2) == 4
assert c2.double(5) == 10
assert c1.double(7) == 14
assert Calculator.double.calls == 3
print("Shared method call count:", Calculator.double.calls)

Shared method call count: 3


### Solution 4: state placement matters

There is one `CountCalls` object in `Calculator.__dict__`. Every instance accesses that same descriptor, so `calls` is shared across all `Calculator` instances. This is sometimes exactly what you want—and sometimes a bug.

# Problem 5 — Maintain a separate counter per object instance

Now change the design so each object gets its own call count.

**Constraint:** do not store the counter directly on the target object under a hard-coded attribute name. Use `WeakKeyDictionary` so the decorator does not unnecessarily keep instances alive.

In [9]:
class PerInstanceCount:
    def __init__(self, fn):
        self.fn = fn
        self._counts = WeakKeyDictionary()
        update_wrapper(self, fn)

    def __call__(self, instance, *args, **kwargs):
        self._counts[instance] = self._counts.get(instance, 0) + 1
        return self.fn(instance, *args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def count_for(self, instance):
        return self._counts.get(instance, 0)

class Worker:
    @PerInstanceCount
    def work(self, units=1):
        return units * 10

w1 = Worker()
w2 = Worker()
w1.work(); w1.work(); w2.work()

descriptor = Worker.work
assert descriptor.count_for(w1) == 2
assert descriptor.count_for(w2) == 1
print("w1:", descriptor.count_for(w1), "w2:", descriptor.count_for(w2))

w1: 2 w2: 1


### Solution 5: why weak references are useful

A normal dictionary keyed by instances would strongly reference those instances and could keep them alive solely because the decorator remembers them. `WeakKeyDictionary` removes entries automatically when keys are garbage-collected, assuming the instances support weak references.

# Problem 6 — Build a parameterized decorator class

Implement this syntax:

```python
@PrefixLogger(prefix="AUDIT")
def transfer(...): ...
```

Here the decorator class is instantiated **before** it receives the function. That means the object must support two phases:

1. configuration
2. decoration

A clean design is to make the configured object receive the function and return a second callable wrapper object.

In [10]:
class _PrefixedCall:
    def __init__(self, fn, prefix):
        self.fn = fn
        self.prefix = prefix
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        print(f"[{self.prefix}] {self.__name__}")
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

class PrefixLogger:
    def __init__(self, prefix="LOG"):
        self.prefix = prefix

    def __call__(self, fn):
        return _PrefixedCall(fn, self.prefix)

@PrefixLogger(prefix="AUDIT")
def transfer(amount):
    return f"transferred {amount}"

assert transfer(50) == "transferred 50"
print(transfer.__name__)

[AUDIT] transfer
transfer


### Solution 6: avoid overloaded state when possible

It is possible to make one class handle both “configuration mode” and “runtime wrapper mode,” but that usually complicates `__call__` and makes debugging harder. Separating the decorator factory object from the callable wrapper object keeps responsibilities clearer.

# Problem 7 — Support instance methods with a parameterized decorator

Use the `PrefixLogger` from the previous problem on an instance method and prove that binding still works.

In [11]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    @PrefixLogger(prefix="BANK")
    def deposit(self, amount):
        self.balance += amount
        return self.balance

acct = BankAccount("Lin", 100)
assert acct.deposit(25) == 125
assert acct.deposit(5) == 130
print("Balance:", acct.balance)

[BANK] deposit
[BANK] deposit
Balance: 130


# Problem 8 — Understand decorator order with class and static methods

Class and static methods are themselves descriptor objects. Decorator order changes **what object gets wrapped**.

Use the safe ordering below:

```python
@classmethod
@Logged

def cm(cls): ...
```

and

```python
@staticmethod
@Logged

def sm(): ...
```

Then inspect the class dictionary to see what is actually stored.

In [12]:
class DemoMethods:
    @classmethod
    @Logged
    def cls_name(cls):
        return cls.__name__

    @staticmethod
    @Logged
    def multiply(a, b):
        return a * b

assert DemoMethods.cls_name() == "DemoMethods"
assert DemoMethods.multiply(6, 7) == 42

print("Stored cls_name:", type(DemoMethods.__dict__["cls_name"]).__name__)
print("Stored multiply:", type(DemoMethods.__dict__["multiply"]).__name__)

Calling cls_name
Calling multiply
Stored cls_name: classmethod
Stored multiply: staticmethod


### Solution 8

With `@classmethod` outermost, `Logged` first receives the plain function and produces a callable descriptor-aware object. Then `classmethod(...)` wraps that result. The same idea applies to `staticmethod`.

The reverse order gives your decorator a `classmethod` or `staticmethod` object instead of the original function, which requires a different implementation strategy.

# Problem 9 — Create a timing decorator with structured statistics

Build a decorator class that tracks:

- number of calls
- total elapsed time
- fastest call
- slowest call
- average duration

Keep the implementation deterministic to test by allowing an injectable clock.

In [13]:
class Timed:
    def __init__(self, fn=None, *, clock=perf_counter):
        self.fn = fn
        self.clock = clock
        self.calls = 0
        self.total = 0.0
        self.minimum = None
        self.maximum = None
        if fn is not None:
            update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        if self.fn is None:
            fn = args[0]
            return type(self)(fn, clock=self.clock)

        start = self.clock()
        try:
            return self.fn(*args, **kwargs)
        finally:
            elapsed = self.clock() - start
            self.calls += 1
            self.total += elapsed
            self.minimum = elapsed if self.minimum is None else min(self.minimum, elapsed)
            self.maximum = elapsed if self.maximum is None else max(self.maximum, elapsed)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    @property
    def average(self):
        return self.total / self.calls if self.calls else 0.0

# deterministic fake clock
class FakeClock:
    def __init__(self, values):
        self._it = iter(values)
    def __call__(self):
        return next(self._it)

fake = FakeClock([0.0, 0.2, 1.0, 1.5, 2.0, 2.1])

@Timed(clock=fake)
def identity(x):
    return x

assert identity(1) == 1
assert identity(2) == 2
assert identity(3) == 3
assert identity.calls == 3
assert abs(identity.total - 0.8) < 1e-12
assert abs(identity.minimum - 0.1) < 1e-12
assert abs(identity.maximum - 0.5) < 1e-12
print("avg:", identity.average)

avg: 0.26666666666666666


### Solution 9: testability best practice

Time-based decorators are hard to test if they always call a real clock. Dependency injection makes behavior reproducible and avoids flaky tests.

# Problem 10 — Exception-aware decorator with `finally`

Create a decorator that records successes and failures without swallowing exceptions.

**Requirements**

- Increment `calls` on every invocation.
- Increment `successes` only if the wrapped function returns normally.
- Increment `failures` when an exception escapes.
- Re-raise the original exception unchanged.

In [14]:
class OutcomeStats:
    def __init__(self, fn):
        self.fn = fn
        self.calls = 0
        self.successes = 0
        self.failures = 0
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        self.calls += 1
        try:
            result = self.fn(*args, **kwargs)
        except Exception:
            self.failures += 1
            raise
        else:
            self.successes += 1
            return result

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

@OutcomeStats
def reciprocal(x):
    return 1 / x

assert reciprocal(4) == 0.25
try:
    reciprocal(0)
except ZeroDivisionError:
    pass

assert (reciprocal.calls, reciprocal.successes, reciprocal.failures) == (2, 1, 1)
print(reciprocal.calls, reciprocal.successes, reciprocal.failures)

2 1 1


# Problem 11 — Implement a configurable retry decorator

Build a decorator with this usage:

```python
@Retry(attempts=3, exceptions=(ValueError,))
def flaky(): ...
```

Best practices:

- validate configuration early
- retry only specified exception types
- re-raise the final failure
- expose the number of actual invocations for testing

In [15]:
class _RetryWrapper:
    def __init__(self, fn, attempts, exceptions):
        self.fn = fn
        self.attempts = attempts
        self.exceptions = exceptions
        self.invocations = 0
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        for attempt in range(1, self.attempts + 1):
            self.invocations += 1
            try:
                return self.fn(*args, **kwargs)
            except self.exceptions:
                if attempt == self.attempts:
                    raise

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

class Retry:
    def __init__(self, *, attempts=3, exceptions=(Exception,)):
        if not isinstance(attempts, int) or attempts < 1:
            raise ValueError("attempts must be a positive integer")
        if not isinstance(exceptions, tuple) or not exceptions:
            raise TypeError("exceptions must be a non-empty tuple")
        self.attempts = attempts
        self.exceptions = exceptions

    def __call__(self, fn):
        return _RetryWrapper(fn, self.attempts, self.exceptions)

state = {"count": 0}

@Retry(attempts=4, exceptions=(ValueError,))
def eventually_works():
    state["count"] += 1
    if state["count"] < 3:
        raise ValueError("not yet")
    return "ok"

assert eventually_works() == "ok"
assert eventually_works.invocations == 3
print("invocations:", eventually_works.invocations)

invocations: 3


# Problem 12 — Add a bounded LRU cache to a decorator class

Implement a small LRU cache without using `functools.lru_cache`.

**Requirements**

- cache by positional and keyword arguments
- distinguish keyword order only by meaning, not call-site order
- evict the least-recently-used entry when capacity is exceeded
- expose hit/miss counters
- support methods through `__get__`

For simplicity, assume all arguments used in the key are hashable.

In [16]:
class LRUCache:
    def __init__(self, fn=None, *, maxsize=128):
        if maxsize < 1:
            raise ValueError("maxsize must be >= 1")
        self.fn = fn
        self.maxsize = maxsize
        self.cache = OrderedDict()
        self.hits = 0
        self.misses = 0
        self._lock = RLock()
        if fn is not None:
            update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        if self.fn is None:
            fn = args[0]
            return type(self)(fn, maxsize=self.maxsize)

        key = (args, tuple(sorted(kwargs.items())))
        with self._lock:
            if key in self.cache:
                self.hits += 1
                self.cache.move_to_end(key)
                return self.cache[key]

        result = self.fn(*args, **kwargs)

        with self._lock:
            self.misses += 1
            self.cache[key] = result
            self.cache.move_to_end(key)
            if len(self.cache) > self.maxsize:
                self.cache.popitem(last=False)
        return result

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

calls = {"n": 0}

@LRUCache(maxsize=2)
def combine(a, b=0):
    calls["n"] += 1
    return a + b

assert combine(1, b=2) == 3       # miss
assert combine(1, b=2) == 3       # hit
assert combine(2, b=3) == 5       # miss
assert combine(4, b=5) == 9       # miss, evicts oldest
assert combine(1, b=2) == 3       # miss again
assert combine.hits == 1
assert combine.misses == 4
print("hits:", combine.hits, "misses:", combine.misses)

hits: 1 misses: 4


### Solution 12: concurrency caveat

The lock protects internal cache mutation. Notice that the wrapped function is deliberately called outside the lock to avoid holding the lock during potentially slow work. This means two threads could compute the same missing key concurrently. Preventing duplicate in-flight computation requires a more elaborate design.

# Problem 13 — Re-entrancy guard

A method may accidentally call itself recursively through an invalid path. Build a decorator that rejects **re-entrant calls on the same instance**, while still allowing different instances to call the same method independently.

Use a `WeakKeyDictionary` to track active instances.

In [17]:
class NonReentrant:
    def __init__(self, fn):
        self.fn = fn
        self._active = WeakKeyDictionary()
        update_wrapper(self, fn)

    def __call__(self, instance, *args, **kwargs):
        if self._active.get(instance, False):
            raise RuntimeError(f"re-entrant call to {self.__name__}")
        self._active[instance] = True
        try:
            return self.fn(instance, *args, **kwargs)
        finally:
            self._active.pop(instance, None)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

class Processor:
    @NonReentrant
    def run(self, depth=0):
        if depth:
            return self.run(depth - 1)
        return "done"

proc = Processor()
assert proc.run() == "done"
try:
    proc.run(1)
except RuntimeError as exc:
    print("Expected:", exc)

Expected: re-entrant call to run


# Problem 14 — A descriptor that caches the bound wrapper per instance

Our earlier `__get__` implementation creates a new `MethodType` object each time the attribute is accessed. Usually that is cheap, but it is useful to understand how to cache a bound wrapper.

Use `WeakKeyDictionary` so the cache does not strongly retain instances.

In [18]:
class CachedBinding:
    def __init__(self, fn):
        self.fn = fn
        self._bound = WeakKeyDictionary()
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        bound = self._bound.get(instance)
        if bound is None:
            bound = MethodType(self, instance)
            self._bound[instance] = bound
        return bound

class CachedDemo:
    @CachedBinding
    def value(self):
        return 42

cd = CachedDemo()
first = cd.value
second = cd.value
assert first is second
assert first() == 42
print("Same bound wrapper object:", first is second)

Same bound wrapper object: True


### Solution 14: subtle reference-cycle caveat

A `WeakKeyDictionary` has weak keys, but the cached **value** here is a bound method that strongly references the instance through `__self__`. That can undermine the intended weak-reference behavior because the value keeps the key alive indirectly.

So this implementation is educational, not ideal for memory-sensitive production code. A better production design would cache only data that does not strongly reference the key, or avoid caching bound method objects entirely.

# Problem 15 — Decorator stacking and `__wrapped__`

Create two class-based decorators and stack them. Confirm that metadata preservation lets `inspect.signature` still recover the original call signature.

In [19]:
class Trace:
    def __init__(self, fn):
        self.fn = fn
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        print(f"TRACE {self.__name__}: args={args}, kwargs={kwargs}")
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

class ValidatePositive:
    def __init__(self, fn):
        self.fn = fn
        update_wrapper(self, fn)

    def __call__(self, x, *args, **kwargs):
        if x <= 0:
            raise ValueError("x must be positive")
        return self.fn(x, *args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

@Trace
@ValidatePositive
def square(x: int) -> int:
    """Square a positive integer."""
    return x * x

assert square(5) == 25
assert square.__name__ == "square"
assert str(signature(square)) == "(x: 'int') -> 'int'"
print(signature(square))

TRACE square: args=(5,), kwargs={}
(x: 'int') -> 'int'


# Problem 16 — Write a decorator that can be disabled without changing call sites

Implement a descriptor-aware decorator class with an `enabled` flag. When disabled, it should call the underlying function directly without logging.

This is useful when instrumentation needs to be toggled at runtime.

In [20]:
class ToggleLogger:
    def __init__(self, fn):
        self.fn = fn
        self.enabled = True
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        if self.enabled:
            print(f"LOG {self.__name__}")
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

class Service:
    @ToggleLogger
    def ping(self, value):
        return value

s = Service()
assert s.ping("a") == "a"
Service.ping.enabled = False
assert s.ping("b") == "b"
Service.ping.enabled = True
print("enabled:", Service.ping.enabled)

LOG ping
enabled: True


# Problem 17 — Validate arguments using the wrapped function's signature

Build a decorator that uses `inspect.signature(...).bind(...)` to validate Python's argument binding explicitly before executing the function.

This is not necessary for ordinary calls—Python already validates arguments—but it is a useful exercise in wrapper introspection and can be extended to custom validation systems.

In [21]:
class BindChecked:
    def __init__(self, fn):
        self.fn = fn
        self.sig = signature(fn)
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        bound = self.sig.bind(*args, **kwargs)
        bound.apply_defaults()
        print("Bound arguments:", dict(bound.arguments))
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

@BindChecked
def format_point(x, y=0, *, label="P"):
    return f"{label}({x}, {y})"

assert format_point(2, label="A") == "A(2, 0)"
try:
    format_point()
except TypeError as exc:
    print("Expected bind error:", exc)

Bound arguments: {'x': 2, 'y': 0, 'label': 'A'}
Expected bind error: missing a required argument: 'x'


# Problem 18 — Capstone: production-style audit decorator

Combine several ideas into one decorator class.

**Requirements**

- works for functions and instance methods
- preserves metadata
- counts total calls, successes, and failures
- records the latest argument binding using the original signature
- does not swallow exceptions
- supports enable/disable of printed audit output
- exposes a `stats()` method

In [22]:
class Audit:
    def __init__(self, fn):
        self.fn = fn
        self.sig = signature(fn)
        self.enabled = True
        self.calls = 0
        self.successes = 0
        self.failures = 0
        self.last_bound_arguments = None
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        self.calls += 1
        try:
            bound = self.sig.bind(*args, **kwargs)
            bound.apply_defaults()
            self.last_bound_arguments = dict(bound.arguments)
            if self.enabled:
                print(f"AUDIT {self.__name__}: {self.last_bound_arguments}")
            result = self.fn(*args, **kwargs)
        except Exception:
            self.failures += 1
            raise
        else:
            self.successes += 1
            return result

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)

    def stats(self):
        return {
            "calls": self.calls,
            "successes": self.successes,
            "failures": self.failures,
            "last_bound_arguments": self.last_bound_arguments,
        }

class Inventory:
    def __init__(self, stock):
        self.stock = stock

    @Audit
    def remove(self, amount, *, allow_negative=False):
        if amount < 0:
            raise ValueError("amount must be non-negative")
        if not allow_negative and amount > self.stock:
            raise ValueError("insufficient stock")
        self.stock -= amount
        return self.stock

inv = Inventory(10)
assert inv.remove(3) == 7
try:
    inv.remove(99)
except ValueError:
    pass

stats = Inventory.remove.stats()
assert stats["calls"] == 2
assert stats["successes"] == 1
assert stats["failures"] == 1
print(stats)

AUDIT remove: {'self': <__main__.Inventory object at 0x000001AB6A4C0D70>, 'amount': 3, 'allow_negative': False}
AUDIT remove: {'self': <__main__.Inventory object at 0x000001AB6A4C0D70>, 'amount': 99, 'allow_negative': False}
{'calls': 2, 'successes': 1, 'failures': 1, 'last_bound_arguments': {'self': <__main__.Inventory object at 0x000001AB6A4C0D70>, 'amount': 99, 'allow_negative': False}}


# Extra challenge set — solve before revealing your own implementation

These are intentionally left as prompts so you can extend the notebook after studying the solved problems above.

1. **TTL cache decorator** — cache values for N seconds; inject a clock for deterministic tests.
2. **Circuit breaker** — after N consecutive failures, reject calls until manually reset.
3. **Per-instance LRU method cache** — prevent results for one object instance from leaking into another.
4. **Access-control decorator** — call a user-supplied predicate before executing the function.
5. **Deprecated-call decorator** — emit a warning only once per decorated function.
6. **Recursive call depth tracker** — expose current and maximum observed depth.
7. **Thread-safe call counter** — make counter increments atomic with a lock.
8. **Descriptor-aware memoized property-like callable** — compare it with `property` and `cached_property`.
9. **Decorator order experiment** — intentionally reverse `@classmethod` and a class-based decorator, inspect the received object, and adapt your implementation.
10. **Custom bound-wrapper object** — implement binding without `MethodType` by returning your own object that stores the instance and forwards calls.

# Reference patterns

## Minimal descriptor-aware decorator class

```python
from functools import update_wrapper
from types import MethodType

class Decorator:
    def __init__(self, fn):
        self.fn = fn
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        return self.fn(*args, **kwargs)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self, instance)
```

## Key design questions

Before writing a stateful decorator class, decide:

- Is state shared per decorated function or separate per target instance?
- Must method binding work?
- Must metadata/signatures be preserved?
- Is the decorator parameterized?
- Is the state thread-safe?
- Can cached state accidentally keep objects alive?
- What should happen when the wrapped function raises?
- Does decorator order matter with descriptors such as `classmethod`, `staticmethod`, `property`, or another custom descriptor?

# Final checklist

A robust class-based decorator should usually be reviewed for:

- [ ] correct forwarding of `*args` and `**kwargs`
- [ ] `__get__` support when decorating instance methods
- [ ] `update_wrapper` / `__wrapped__`
- [ ] explicit state ownership
- [ ] exception transparency
- [ ] predictable decorator ordering
- [ ] memory behavior of caches and instance maps
- [ ] concurrency behavior if mutable shared state exists
- [ ] testability through dependency injection for clocks, delays, random sources, etc.
- [ ] focused tests for plain functions, instance methods, class access, and stacked decorators